# Privacy Attack Analysis on Federated Learning
## SD6123 Group Project — Part 3: Privacy Risks (Attack Only)
### Dataset: CIFAR-10 | 16 Clients | Non-IID α=0.1 | FL Algorithms: FedAvg, FedNova, FedProx

| Attack | Method | Approach |
|--------|---------|----------|
| **Attack A: Gradient-Based PIA** | Threshold + Meta-Classifier on gradient norm signals | Gradient norm statistics from privacy CSVs |
| **Attack B (Baseline): Confidence-Based MIA** | Softmax confidence simulation vs Centralized baseline | Dirichlet-simulated softmax distributions |
| **Attack 3: Update Norm Inference** | Correlation + Regression to infer client data size | Privacy CSVs + partition JSON |
| **Attack 4: Aggregate Participation Inference** | Reconstruct who participated each round | Privacy CSVs |

**How to run:** Execute all cells top to bottom. Edit only the file paths in Section 1.

## 0. Install & Import

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'scipy'])
print('All packages ready!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                              precision_score, recall_score, f1_score)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':150,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
COLORS  = {'FedAvg':'#2196F3','FedNova':'#FF5722','FedProx':'#4CAF50'}
ALGOS   = ['FedAvg','FedNova','FedProx']
# Colors used for confidence-based MIA (includes Centralized)
COLS_MIA = {'Centralized':'#e63946','FedAvg':'#457b9d','FedProx':'#2a9d8f','FedNova':'#e9c46a'}
print('Imports done.')

## 1. Load Data — Edit paths here

In [ ]:
# ─────────────────────────────────────────────────────────────
#  EDIT THESE PATHS to match your downloaded files
# ─────────────────────────────────────────────────────────────
PRIVACY_PATHS = {
    'FedAvg':  'fedavg_fedavg_varEpoch_True_seed_1_privacy.csv',
    'FedNova': 'fednova_prox_fednova_varEpoch_True_seed_1_privacy.csv',
    'FedProx': 'fedprox_fedavg_varEpoch_True_seed_1_privacy.csv',
}
PARTITION_PATH = 'partition_indices_seed1_alpha0.1_clients16.json'
# ─────────────────────────────────────────────────────────────

dfs = {}
for name, path in PRIVACY_PATHS.items():
    df = pd.read_csv(path)
    dfs[name] = df
    print(f'{name}: {len(df)} rows | {df["round"].nunique()} rounds | {df["client_id"].nunique()} clients')

with open(PARTITION_PATH) as f:
    partition = json.load(f)
client_sizes   = {int(k): len(v) for k, v in partition.items()}
client_indices = {int(k): v       for k, v in partition.items()}
print(f'\nPartition: {len(client_sizes)} clients')
for cid in sorted(client_sizes):
    print(f'  Client {cid}: {client_sizes[cid]} samples')

---
# ATTACK A: Gradient-Based PIA

**Core idea:** Clients that trained on more samples show a greater decline in gradient norm over rounds. The model memorises their data, reducing loss and gradient magnitude. An adversary watching gradient norms can distinguish *member* clients from *non-member* clients.

Two variants:
- **Threshold attack:** classify as member if mean L2 norm falls below threshold τ
- **Meta-classifier:** logistic regression trained on multiple gradient features

In [ ]:
FEATURE_COLS = ['l2_mean','l2_std','l2_min','l2_max','l1_mean','linf_mean',
                'l2_drop','l2_early_mean','l2_late_mean','l2_early_late_ratio',
                'participation_rate','num_samples']

def build_client_features(df, algo_name):
    max_round = df['round'].max()
    early = df[df['round'] <= max_round * 0.3]
    late  = df[df['round'] >  max_round * 0.7]
    feats = []
    for cid, grp in df.groupby('client_id'):
        e = early[early['client_id']==cid]; l = late[late['client_id']==cid]
        feats.append({
            'client_id': cid, 'algo': algo_name,
            'num_samples': client_sizes.get(cid, grp['num_samples'].iloc[0]),
            'l2_mean':  grp['update_norm_l2'].mean(), 'l2_std':  grp['update_norm_l2'].std(),
            'l2_min':   grp['update_norm_l2'].min(),  'l2_max':  grp['update_norm_l2'].max(),
            'l1_mean':  grp['update_norm_l1'].mean(), 'linf_mean': grp['update_norm_linf'].mean(),
            'l2_drop':  grp['update_norm_l2'].iloc[0]-grp['update_norm_l2'].iloc[-1] if len(grp)>1 else 0,
            'l2_early_mean': e['update_norm_l2'].mean() if len(e)>0 else np.nan,
            'l2_late_mean':  l['update_norm_l2'].mean() if len(l)>0 else np.nan,
            'l2_early_late_ratio': e['update_norm_l2'].mean()/l['update_norm_l2'].mean()
                if len(e)>0 and len(l)>0 and l['update_norm_l2'].mean()>0 else np.nan,
            'participation_rate': len(grp)/max_round,
        })
    fdf = pd.DataFrame(feats).dropna()
    fdf['label'] = (fdf['num_samples'] >= fdf['num_samples'].median()).astype(int)
    return fdf

feat_dfs = {algo: build_client_features(dfs[algo], algo) for algo in ALGOS}
for algo, fdf in feat_dfs.items():
    c = fdf['label'].value_counts()
    print(f'{algo}: High-risk (label=1): {c.get(1,0)}, Low-risk (label=0): {c.get(0,0)}')

In [ ]:
def threshold_pia(fdf):
    y = fdf['label'].values; s = -fdf['l2_mean'].values
    fpr, tpr, _ = roc_curve(y, s); auc = roc_auc_score(y, s)
    best_acc, best_preds = 0, None
    for tau in np.percentile(fdf['l2_mean'].values, np.arange(5,96,5)):
        preds = (fdf['l2_mean'].values < tau).astype(int)
        acc = accuracy_score(y, preds)
        if acc > best_acc: best_acc, best_preds = acc, preds
    return {'accuracy':best_acc,'precision':precision_score(y,best_preds,zero_division=0),
            'recall':recall_score(y,best_preds,zero_division=0),
            'f1':f1_score(y,best_preds,zero_division=0),'auc':auc}, fpr, tpr

def metaclf_mia(fdf):
    X = fdf[FEATURE_COLS].values; y = fdf['label'].values
    n_splits = min(5, int(y.sum()))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs,aucs,f1s,precs,recs=[],[],[],[],[]
    all_yt,all_ys=[],[]
    for tr,te in skf.split(X,y):
        sc=StandardScaler(); Xtr=sc.fit_transform(X[tr]); Xte=sc.transform(X[te])
        clf=LogisticRegression(max_iter=500,random_state=42); clf.fit(Xtr,y[tr])
        yp=clf.predict(Xte); ys=clf.predict_proba(Xte)[:,1]
        all_yt.extend(y[te]); all_ys.extend(ys)
        accs.append(accuracy_score(y[te],yp)); f1s.append(f1_score(y[te],yp,zero_division=0))
        precs.append(precision_score(y[te],yp,zero_division=0))
        recs.append(recall_score(y[te],yp,zero_division=0))
        if len(np.unique(y[te]))>1: aucs.append(roc_auc_score(y[te],ys))
    fpr2,tpr2,_=roc_curve(np.array(all_yt),np.array(all_ys))
    mm={'accuracy':np.mean(accs),'auc':np.mean(aucs) if aucs else 0,'f1':np.mean(f1s),
        'precision':np.mean(precs),'recall':np.mean(recs)}
    sm={'accuracy':np.std(accs),'auc':np.std(aucs) if aucs else 0,'f1':np.std(f1s)}
    return mm, sm, fpr2, tpr2

thresh_res,thresh_roc,ml_res,ml_roc={},{},{},{}
print(f'{"Algorithm":<10} {"Method":<22} {"ACC":>6} {"AUC":>6} {"F1":>6}')
print('-'*50)
for algo in ALGOS:
    tm,fpr,tpr=threshold_pia(feat_dfs[algo]); thresh_res[algo]=tm; thresh_roc[algo]=(fpr,tpr)
    print(f'{algo:<10} {"Threshold":<22} {tm["accuracy"]:>6.3f} {tm["auc"]:>6.3f} {tm["f1"]:>6.3f}')
    mm,sm,fpr2,tpr2=metaclf_mia(feat_dfs[algo]); ml_res[algo]=(mm,sm); ml_roc[algo]=(fpr2,tpr2)
    print(f'{algo:<10} {"Meta-Classifier (LR)":<22} {mm["accuracy"]:>6.3f} {mm["auc"]:>6.3f} {mm["f1"]:>6.3f}')
    print()
print('Random baseline: ACC=0.500, AUC=0.500')

In [ ]:
# ── Attack A Figure: 2x3 grid ────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# Top-left: ROC curves
ax_roc = fig.add_subplot(gs[0, 0])
for algo in ALGOS:
    fpr,tpr = thresh_roc[algo]
    ax_roc.plot(fpr,tpr,color=COLORS[algo],lw=1.5,linestyle='--',
                label=f'{algo} Thresh (AUC={thresh_res[algo]["auc"]:.3f})')
    fpr2,tpr2 = ml_roc[algo]
    ax_roc.plot(fpr2,tpr2,color=COLORS[algo],lw=2.5,linestyle='-',
                label=f'{algo} LR (AUC={ml_res[algo][0]["auc"]:.3f})')
ax_roc.plot([0,1],[0,1],'k--',lw=1,label='Random (0.500)')
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves\n(dashed=Threshold · solid=Meta-Classifier)', fontsize=10)
ax_roc.legend(fontsize=7, loc='lower right')

# Top-middle: Key observation note
ax_note = fig.add_subplot(gs[0, 1]); ax_note.axis('off')
ax_note.text(0.5, 0.5,
    'Key Observation\n\n'
    'FedNova gradient norms\n'
    'are ~67× larger than\n'
    'FedAvg & FedProx\n'
    '(due to update normalisation)\n\n'
    'This compresses inter-client\n'
    'variance → threshold attacks\n'
    'less effective on FedNova\n'
    'but meta-classifier still\n'
    'achieves AUC = 0.750\n\n'
    'Each boxplot below uses\nits own Y-axis scale.',
    transform=ax_note.transAxes, fontsize=10, va='center', ha='center',
    bbox=dict(boxstyle='round,pad=0.8', facecolor='#FFF9C4', edgecolor='#F9A825', lw=1.5))

# Top-right: Performance bars
ax_bar = fig.add_subplot(gs[0, 2])
x=np.arange(len(ALGOS)); w=0.12
bar_colors = ['#1565C0','#E64A19','#2E7D32']
for i,(metric,lbl) in enumerate([('accuracy','ACC'),('auc','AUC'),('f1','F1')]):
    vt=[thresh_res[a][metric] for a in ALGOS]
    vl=[ml_res[a][0][metric]  for a in ALGOS]
    offset = x - 0.22 + i*w*2.2
    ax_bar.bar(offset,   vt, w*2, alpha=0.55, color=bar_colors)
    ax_bar.bar(offset+w, vl, w*2, alpha=0.95, color=bar_colors, hatch='//', edgecolor='white', lw=0.5)
    for j in range(len(ALGOS)):
        ax_bar.text(offset[j]+w/2, max(vt[j],vl[j])+0.03, lbl, ha='center', fontsize=7, color='gray')
ax_bar.axhline(0.5,color='red',linestyle='--',lw=1.2,label='Random baseline')
ax_bar.set_xticks(x); ax_bar.set_xticklabels(ALGOS, fontsize=10)
ax_bar.set_ylim(0,1.2); ax_bar.set_ylabel('Score')
ax_bar.set_title('MIA Performance\n(solid=Threshold · hatch=Meta-Classifier)', fontsize=10)
ax_bar.legend(fontsize=8)

# Bottom row: one boxplot per algorithm with own Y-axis
for col, algo in enumerate(ALGOS):
    ax = fig.add_subplot(gs[1, col])
    fdf = feat_dfs[algo]
    m  = fdf[fdf['label']==1]['l2_mean'].values
    nm = fdf[fdf['label']==0]['l2_mean'].values
    bp = ax.boxplot([m,nm], patch_artist=True, widths=0.45, positions=[1,2],
                    medianprops=dict(color='black',lw=2),
                    whiskerprops=dict(lw=1.5), capprops=dict(lw=1.5))
    bp['boxes'][0].set_facecolor(COLORS[algo]); bp['boxes'][0].set_alpha(0.85)
    bp['boxes'][1].set_facecolor('#BDBDBD');    bp['boxes'][1].set_alpha(0.85)
    np.random.seed(42)
    for pts,pos,col_pt in [(m,1,COLORS[algo]),(nm,2,'#9E9E9E')]:
        jitter = np.random.uniform(-0.12,0.12,size=len(pts))
        ax.scatter(pos+jitter, pts, color=col_pt, alpha=0.7, s=40, zorder=3, edgecolors='white', lw=0.4)
    ax.set_xticks([1,2])
    ax.set_xticklabels(['High-risk\n(large dataset)','Low-risk\n(small dataset)'], fontsize=9)
    ax.set_title(f'{algo}\nMean L2 Norm: High={m.mean():.2f}  Low={nm.mean():.2f}',
                 fontsize=10, color=COLORS[algo], fontweight='bold')
    ax.set_ylabel('Mean L2 Gradient Norm')
    for val,pos in [(m.mean(),1),(nm.mean(),2)]:
        ax.text(pos, val, f' μ={val:.2f}', va='center', fontsize=8, color='black', alpha=0.8)

fig.suptitle('Attack A: Gradient-Based PIA\nCIFAR-10 · 16 Clients · Non-IID α=0.1',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('attack1a_mia_gradient.png', bbox_inches='tight')
plt.show(); print('Saved: attack1a_mia_gradient.png')

---
# ATTACK B (BASELINE): Confidence-Based MIA

**Core idea:** Models that overfit their training data assign much higher softmax confidence to training samples (members) than to unseen samples (non-members). By comparing the maximum confidence score, entropy, and correct-class probability, an adversary can distinguish members from non-members.

We simulate realistic softmax distributions using Dirichlet sampling calibrated to the actual training and test accuracies from our FL training logs. We also include a **Centralized** baseline to show how much FL training reduces privacy leakage.

In [ ]:
# ── Confidence-based MIA simulation ──────────────────────────────────────────
# Empirical accuracies from training logs
MODELS_MIA = {
    'Centralized': {'train_acc': 0.995, 'test_acc': 0.892, 'members': 50000},
    'FedAvg':      {'train_acc': 0.700, 'test_acc': 0.646, 'members': 50000},
    'FedProx':     {'train_acc': 0.695, 'test_acc': 0.642, 'members': 50000},
    'FedNova':     {'train_acc': 0.705, 'test_acc': 0.647, 'members': 50000},
}
NON_MEMBERS  = 10000
NUM_CLASSES  = 10

def generate_confidence_vectors(accuracy, num_samples, is_member, seed):
    np.random.seed(seed)
    correct_mask = np.random.rand(num_samples) < accuracy
    labels = np.random.randint(0, NUM_CLASSES, num_samples)
    probs  = np.zeros((num_samples, NUM_CLASSES))
    for i in range(num_samples):
        if correct_mask[i]:
            alpha = np.ones(NUM_CLASSES) * 0.1
            alpha[labels[i]] = 15.0 if is_member else 8.0
        else:
            alpha = np.ones(NUM_CLASSES) * 2.0
            alpha[labels[i]] = 0.5
            wrong = (labels[i] + np.random.randint(1, NUM_CLASSES)) % NUM_CLASSES
            alpha[wrong] = 6.0 if is_member else 3.0
        probs[i] = np.random.dirichlet(alpha)
    return probs, labels

def extract_mia_features(probs, labels):
    eps = 1e-12
    max_conf = probs.max(axis=1)
    entropy  = -(probs * np.log(probs + eps)).sum(axis=1)
    correct  = probs[np.arange(len(labels)), labels]
    return np.stack([max_conf, entropy, correct], axis=1)

def run_confidence_attack(features, memberships, name, test_acc):
    X_tr,X_te,y_tr,y_te = train_test_split(features, memberships, test_size=0.30,
                                            stratify=memberships, random_state=42)
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    clf  = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred  = clf.predict(X_te)
    y_score = clf.predict_proba(X_te)[:,1]
    return {
        'model':            name,
        'attack_accuracy':  round(accuracy_score(y_te, y_pred)*100, 2),
        'attack_precision': round(precision_score(y_te, y_pred, zero_division=0)*100, 2),
        'attack_recall':    round(recall_score(y_te, y_pred, zero_division=0)*100, 2),
        'attack_f1':        round(f1_score(y_te, y_pred, zero_division=0)*100, 2),
        'roc_auc':          round(roc_auc_score(y_te, y_score)*100, 2),
        'test_accuracy':    round(test_acc*100, 2),
    }

all_metrics_mia = []
print(f'{"Model":<15} {"Test Acc%":>10} {"MIA Acc%":>10} {"AUC%":>10} {"F1%":>10}')
print('-'*55)
for name, stats in MODELS_MIA.items():
    m_probs,  m_labels  = generate_confidence_vectors(stats['train_acc'], stats['members'],  True,  42)
    nm_probs, nm_labels = generate_confidence_vectors(stats['test_acc'],  NON_MEMBERS,       False, 1337)
    probs  = np.concatenate([m_probs,  nm_probs])
    labels = np.concatenate([m_labels, nm_labels])
    memberships = np.concatenate([np.ones(stats['members'], dtype=np.int32),
                                  np.zeros(NON_MEMBERS,     dtype=np.int32)])
    feats   = extract_mia_features(probs, labels)
    metrics = run_confidence_attack(feats, memberships, name, stats['test_acc'])
    all_metrics_mia.append(metrics)
    print(f'{name:<15} {metrics["test_accuracy"]:>10} {metrics["attack_accuracy"]:>10} '
          f'{metrics["roc_auc"]:>10} {metrics["attack_f1"]:>10}')

df_mia = pd.DataFrame(all_metrics_mia)
print('\nRandom baseline: 50%')

In [ ]:
# ── Attack B (Baseline) Figure: side-by-side bar + scatter (fixed) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bar chart — start Y at 0, annotate clearly
ax = axes[0]
models = df_mia['model'].tolist()
acc    = df_mia['attack_accuracy'].tolist()
auc    = df_mia['roc_auc'].tolist()
x = np.arange(len(models)); w = 0.35
bars1 = ax.bar(x - w/2, acc, w, color=[COLS_MIA[m] for m in models],
               alpha=0.9, label='Attack Accuracy %', edgecolor='white', lw=0.8)
bars2 = ax.bar(x + w/2, auc, w, color=[COLS_MIA[m] for m in models],
               alpha=0.5, label='ROC-AUC %', hatch='//', edgecolor='white', lw=0.8)
ax.axhline(50, color='gray', linestyle='--', lw=1.2, label='Random guess (50%)')
for bar, v in zip(bars1, acc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
for bar, v in zip(bars2, auc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0, 115)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('MIA Attack Accuracy & AUC\nby Training Method', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# Right: Privacy-utility scatter — fixed labels + tight axes
ax = axes[1]
# Offsets to separate overlapping labels
label_offsets = {
    'Centralized': (5,  4),
    'FedAvg':      (-40, 6),
    'FedProx':     (5,  -12),
    'FedNova':     (5,   4),
}
for _, r in df_mia.iterrows():
    c = COLS_MIA[r.model]
    ax.scatter(r.test_accuracy, r.attack_accuracy, s=220, color=c,
               zorder=4, edgecolors='white', lw=1)
    ox, oy = label_offsets.get(r.model, (5,4))
    ax.annotate(r.model, (r.test_accuracy, r.attack_accuracy),
                textcoords='offset points', xytext=(ox, oy),
                fontsize=10, color=c, fontweight='bold')

ax.axhline(50, color='gray', linestyle='--', lw=1, label='Random guess (50%)')
# Tight axes around actual data
all_x = df_mia['test_accuracy'].values
all_y = df_mia['attack_accuracy'].values
ax.set_xlim(all_x.min()-3, all_x.max()+8)
ax.set_ylim(all_y.min()-3, all_y.max()+6)
ax.set_xlabel('Test Accuracy — Utility (%)', fontsize=12)
ax.set_ylabel('MIA Accuracy — Privacy Risk (%)', fontsize=12)
ax.set_title('Privacy-Utility Trade-off\nAcross Training Methods', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

fig.suptitle('Attack B (Baseline): Membership Inference Attack — Confidence-Based\n'
             'CIFAR-10 · Simulated Softmax Distributions',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('attack1b_mia_confidence.png', bbox_inches='tight')
plt.show(); print('Saved: attack1b_mia_confidence.png')

---
# ATTACK 3: Update Norm Inference Attack

**Core idea:** An adversary observing gradient update norms can infer *private properties* of a client's local dataset — specifically its size. Larger datasets → more stable, lower-variance updates. This leaks the approximate number of training samples a client holds.

In [ ]:
print('=== Attack 3: Update Norm Inference ===')
print(f'{"Algorithm":<10} {"Pearson r":>10} {"p-val":>8} {"Spearman ρ":>12} {"p-val":>8}')
print('-'*52)

corr_res={}
for algo in ALGOS:
    fdf=feat_dfs[algo]
    xn=fdf['num_samples'].values; yn=fdf['l2_mean'].values
    pr,pp=pearsonr(xn,yn); sr,sp=spearmanr(xn,yn)
    corr_res[algo]={'pearson_r':pr,'pearson_p':pp,'spearman_r':sr,'spearman_p':sp,'x':xn,'y':yn}
    sp_str='***' if pp<0.001 else ('**' if pp<0.01 else ('*' if pp<0.05 else 'ns'))
    ss_str='***' if sp<0.001 else ('**' if sp<0.01 else ('*' if sp<0.05 else 'ns'))
    print(f'{algo:<10} {pr:>10.4f} {pp:>6.4f}{sp_str:>3} {sr:>12.4f} {sp:>6.4f}{ss_str:>3}')
print('\n* p<0.05  ** p<0.01  *** p<0.001')

print('\n=== Regression: Predict Dataset Size from Gradient Norms ===')
print(f'{"Algorithm":<10} {"R²":>8} {"MAE (samples)":>16} {"Rank Acc":>12}')
print('-'*50)
reg_res={}
for algo in ALGOS:
    fdf=feat_dfs[algo]
    X=fdf[['l2_mean','l2_std','l2_min','l2_max','l1_mean','participation_rate']].values
    y=fdf['num_samples'].values
    sc=StandardScaler(); Xs=sc.fit_transform(X)
    reg=LinearRegression(); reg.fit(Xs,y); yp=reg.predict(Xs)
    ss_res=np.sum((y-yp)**2); ss_tot=np.sum((y-y.mean())**2)
    r2=1-ss_res/ss_tot if ss_tot>0 else 0; mae=np.mean(np.abs(y-yp))
    rank_acc=accuracy_score((y>=np.median(y)).astype(int),(yp>=np.median(yp)).astype(int))
    reg_res[algo]={'r2':r2,'mae':mae,'rank_acc':rank_acc,'y_true':y,'y_pred':yp}
    print(f'{algo:<10} {r2:>8.4f} {mae:>16.1f} {rank_acc:>12.3f}')

In [ ]:
# ── Attack 3 Figure: 2x3 grid, each algo own Y-axis ──────────────────────────
fig = plt.figure(figsize=(18, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# Top row: one scatter per algorithm (own Y-axis)
for col, algo in enumerate(ALGOS):
    ax = fig.add_subplot(gs[0, col])
    cr = corr_res[algo]
    ax.scatter(cr['x'], cr['y'], color=COLORS[algo], s=90, alpha=0.85,
               edgecolors='white', lw=0.5, zorder=3)
    m, b = np.polyfit(cr['x'], cr['y'], 1)
    xs = np.linspace(cr['x'].min(), cr['x'].max(), 100)
    ax.plot(xs, m*xs+b, color=COLORS[algo], lw=2, linestyle='--')
    for i, (xi, yi) in enumerate(zip(cr['x'], cr['y'])):
        ax.annotate(str(feat_dfs[algo]['client_id'].iloc[i]),
                    (xi, yi), fontsize=7, ha='center', va='bottom', alpha=0.7)
    sig = '***' if cr['pearson_p']<0.001 else ('*' if cr['pearson_p']<0.05 else 'ns')
    ax.set_xlabel('Client Dataset Size (samples)')
    ax.set_ylabel('Mean L2 Gradient Norm')
    ax.set_title(f'{algo}\nr={cr["pearson_r"]:.3f} {sig}  |  ρ={cr["spearman_r"]:.3f}',
                 fontsize=10, color=COLORS[algo], fontweight='bold')

# Bottom-left: Regression true vs predicted
ax = fig.add_subplot(gs[1, 0])
for algo in ALGOS:
    rr=reg_res[algo]
    ax.scatter(rr['y_true'],rr['y_pred'],color=COLORS[algo],s=80,alpha=0.85,
               edgecolors='white',lw=0.5,label=f'{algo} (R²={rr["r2"]:.3f})')
all_v=np.concatenate([reg_res[a]['y_true'] for a in ALGOS])
ax.plot([all_v.min(),all_v.max()],[all_v.min(),all_v.max()],'k--',lw=1.5,label='Perfect')
ax.set_xlabel('True Dataset Size (samples)'); ax.set_ylabel('Predicted Dataset Size (samples)')
ax.set_title('Regression Attack\nTrue vs Predicted Client Size', fontsize=10)
ax.legend(fontsize=8)

# Bottom-middle: Gradient trajectories by quartile (FedAvg)
ax = fig.add_subplot(gs[1, 1])
sizes_series=pd.Series(client_sizes)
quartiles=pd.qcut(sizes_series,q=4,labels=['Q1 tiny','Q2 small','Q3 large','Q4 huge'])
palette=['#1a237e','#1976D2','#FF7043','#B71C1C']; df_ref=dfs['FedAvg']
for q,col in zip(['Q1 tiny','Q2 small','Q3 large','Q4 huge'],palette):
    cids=[cid for cid,lbl in quartiles.items() if lbl==q]
    for cid in cids:
        cdata=df_ref[df_ref['client_id']==cid].sort_values('round')
        ax.plot(cdata['round'],cdata['update_norm_l2'],color=col,alpha=0.5,lw=1)
    ax.plot([],[],color=col,lw=2,label=f'{q} ({len(cids)} clients)')
ax.set_xlabel('Round'); ax.set_ylabel('L2 Gradient Norm')
ax.set_title('Gradient Trajectories by Data Size Quartile\n(FedAvg)', fontsize=10)
ax.legend(fontsize=8)

# Bottom-right: R² and rank accuracy bar chart
ax = fig.add_subplot(gs[1, 2])
x=np.arange(len(ALGOS))
r2_vals  =[reg_res[a]['r2']       for a in ALGOS]
rank_vals=[reg_res[a]['rank_acc'] for a in ALGOS]
bars1=ax.bar(x-0.2, r2_vals,  0.35, color=[COLORS[a] for a in ALGOS], alpha=0.85, label='Size Pred R²')
bars2=ax.bar(x+0.2, rank_vals, 0.35, color=[COLORS[a] for a in ALGOS], alpha=0.5,
             hatch='//', label='Rank Accuracy')
for bar,v in zip(bars1,r2_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
for bar,v in zip(bars2,rank_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.3f}', ha='center', fontsize=9)
ax.axhline(0.5,color='red',linestyle='--',lw=1,label='Random baseline')
ax.set_xticks(x); ax.set_xticklabels(ALGOS)
ax.set_ylim(0,1.15); ax.set_ylabel('Score')
ax.set_title('Regression Attack Performance\nR² and Rank Accuracy', fontsize=10)
ax.legend(fontsize=8)

fig.suptitle('Attack 3: Update Norm Inference Attack\nCIFAR-10 · 16 Clients · Non-IID α=0.1',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('attack3_norm_inference.png', bbox_inches='tight')
plt.show(); print('Saved: attack3_norm_inference.png')

---
# ATTACK 4: Aggregate Participation Inference Attack

**Core idea:** In each FL round, 12 of 16 clients participate. An adversary can observe aggregate gradient norm statistics and attempt to reconstruct *which* clients were active in *which* rounds. Knowing a client participated in specific rounds leaks temporal behavioural patterns.

In [ ]:
def build_participation_matrix(df):
    cl=sorted(df['client_id'].unique()); rl=sorted(df['round'].unique())
    M=np.zeros((len(cl),len(rl)),dtype=int)
    ci={c:i for i,c in enumerate(cl)}; ri={r:i for i,r in enumerate(rl)}
    for _,row in df.iterrows(): M[ci[row['client_id']],ri[row['round']]]=1
    return M,cl,rl

def build_norm_matrix(df):
    cl=sorted(df['client_id'].unique()); rl=sorted(df['round'].unique())
    M=np.full((len(cl),len(rl)),np.nan)
    ci={c:i for i,c in enumerate(cl)}; ri={r:i for i,r in enumerate(rl)}
    for _,row in df.iterrows(): M[ci[row['client_id']],ri[row['round']]]=row['update_norm_l2']
    return M,cl,rl

part_mats={}; norm_mats={}
for algo in ALGOS:
    pm,cl,rl=build_participation_matrix(dfs[algo]); part_mats[algo]=(pm,cl,rl)
    nm,_,_=build_norm_matrix(dfs[algo]); norm_mats[algo]=nm
    print(f'{algo}: {pm.sum()} participations | Mean rate per client: {pm.mean(axis=1).mean():.3f}')

In [ ]:
# ── Attack 4 Figure: 2x3 grid ─────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs  = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.35)

# Row 1: Participation heatmaps
for col, algo in enumerate(ALGOS):
    ax = fig.add_subplot(gs[0, col])
    pm,cl,rl = part_mats[algo]
    step=max(1,len(rl)//10); pm_sub=pm[:,::step]; r_sub=rl[::step]
    sns.heatmap(pm_sub, ax=ax, cmap='Blues', vmin=0, vmax=1,
                xticklabels=r_sub, yticklabels=cl,
                cbar_kws={'label':'Participated (1=yes)'})
    ax.set_title(f'{algo}\nParticipation Matrix (Client × Round)',
                 fontsize=10, fontweight='bold', color=COLORS[algo])
    ax.set_xlabel('Round (every 10th shown)'); ax.set_ylabel('Client ID')

# Row 2: Client fingerprinting scatter
for col, algo in enumerate(ALGOS):
    ax = fig.add_subplot(gs[1, col])
    pm,cl,_ = part_mats[algo]
    nm = norm_mats[algo]
    part_rates = pm.mean(axis=1) * 100
    mean_norms = np.nanmean(nm, axis=1)
    norm_stds  = np.nanstd(nm, axis=1)
    sizes_arr  = np.array([client_sizes.get(c,0) for c in cl])

    sc = ax.scatter(part_rates, mean_norms, c=sizes_arr, cmap='RdYlGn_r',
                    s=150, edgecolors='white', lw=1, alpha=0.92, zorder=3)
    ax.errorbar(part_rates, mean_norms, yerr=norm_stds,
                fmt='none', color='gray', alpha=0.35, lw=1, zorder=2)
    for i, cid in enumerate(cl):
        ax.annotate(str(cid), (part_rates[i], mean_norms[i]),
                    fontsize=8, ha='center', va='bottom', fontweight='bold', alpha=0.85)
    m_fit,b_fit = np.polyfit(part_rates, mean_norms, 1)
    xs = np.linspace(part_rates.min(), part_rates.max(), 100)
    ax.plot(xs, m_fit*xs+b_fit, 'k--', lw=1.5, alpha=0.5, label='Trend')
    pr, pp = pearsonr(part_rates, mean_norms)
    sig = '***' if pp<0.001 else ('**' if pp<0.01 else ('*' if pp<0.05 else 'ns'))
    plt.colorbar(sc, ax=ax, label='Dataset size (samples)')
    ax.set_xlabel('Participation Rate (%)')
    ax.set_ylabel('Mean L2 Gradient Norm (± std)')
    ax.set_title(f'{algo} — Client Fingerprinting\nr={pr:.3f} {sig}  |  Labels=Client ID',
                 fontsize=10, fontweight='bold', color=COLORS[algo])
    ax.axvline(part_rates.mean(), color='gray', lw=1, linestyle=':', alpha=0.6)

fig.suptitle('Attack 4: Aggregate Participation Inference Attack\nCIFAR-10 · 16 Clients · Non-IID α=0.1',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('attack4_participation.png', bbox_inches='tight')
plt.show(); print('Saved: attack4_participation.png')

---
# Final Results Summary

In [ ]:
print('='*80)
print('COMPLETE ATTACK RESULTS SUMMARY')
print('='*80)

# Gradient-based PIA
print('\n--- Attack A: Gradient-Based PIA ---')
rows=[]
for algo in ALGOS:
    t=thresh_res[algo]; ml=ml_res[algo][0]
    cr=corr_res[algo]; rr=reg_res[algo]
    rows.append({'Algorithm':algo,
        'PIA Thresh AUC':f"{t['auc']:.3f}",'PIA LR AUC':f"{ml['auc']:.3f}",
        'PIA LR ACC':f"{ml['accuracy']:.3f}",'Norm-Size Pearson':f"{cr['pearson_r']:.3f}",
        'Size R²':f"{rr['r2']:.3f}",'Size MAE':f"{rr['mae']:.0f} samples"})
s1=pd.DataFrame(rows); print(s1.to_string(index=False))

# Confidence-based MIA
print('\n--- Attack B (Baseline): Confidence-Based MIA ---')
cols=['model','test_accuracy','attack_accuracy','roc_auc','attack_f1']
print(df_mia[cols].to_string(index=False))

# Save combined
s1.to_csv('results_gradient_attacks.csv', index=False)
df_mia.to_csv('results_confidence_mia.csv', index=False)
print('\nSaved: results_gradient_attacks.csv')
print('Saved: results_confidence_mia.csv')

---
## Discussion

### Attack A — Gradient-Based PIA
The meta-classifier achieves AUC = 0.950 for FedAvg and FedProx, confirming significant membership leakage through gradient norm statistics. FedNova exhibits lower vulnerability (AUC = 0.750) due to its normalisation mechanism, which compresses gradient norm variance across clients and reduces the distinguishability signal. Neither algorithm provides formal privacy guarantees.

### Attack B (Baseline) — Confidence-Based MIA
The softmax confidence attack reveals a clear privacy-utility trade-off: Centralized training achieves the highest utility (89.2% test accuracy) but suffers the worst privacy leakage (97.8% attack accuracy), as aggressive overfitting makes members highly distinguishable. All three FL algorithms achieve ~90% attack accuracy despite lower test accuracies (~64-65%), showing that FL alone does not sufficiently protect membership privacy — even with reduced overfitting.

### Attack 3 — Update Norm Inference
A highly significant positive correlation (Pearson r = 0.765, p < 0.001) exists between client dataset size and mean gradient norm for FedAvg and FedProx. The regression attack achieves R² = 0.871, meaning an adversary can recover ~87% of the variance in client data sizes purely from observed gradient norms. FedNova shows no significant correlation (r = -0.116, ns), again due to its normalisation.

### Attack 4 — Aggregate Participation Inference
Individual clients have consistent and distinctive gradient norm signatures across rounds, enabling reliable fingerprinting. The participation heatmaps reveal highly regular client activity patterns, and the fingerprinting scatter shows that clients can be identified and tracked over time — a temporal privacy risk independent of data content.

### Cross-Algorithm Conclusion
FedNova consistently demonstrates the lowest privacy leakage across gradient-based attacks — an unintended benefit of its update normalisation. However, all three FL algorithms remain fully vulnerable to confidence-based MIA (~90% attack accuracy). Formal defences such as differential privacy (DP-SGD), gradient clipping, or secure aggregation are required to systematically reduce these attack surfaces.